In [1]:
#!pip install langchain_google_genai

In [2]:
import os
from dotenv import load_dotenv

# Load the .env file
load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")

In [3]:
import getpass
import os

#Langchain prompts
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompt_values import ChatPromptValue

#Langchain mensajesTool
from langchain_core.messages import AIMessage

#Langchain cadena
from langchain_core.runnables.base import RunnableSequence

#Langchain chat model (Google)
from langchain_google_genai import ChatGoogleGenerativeAI

#Langsmith
from langsmith import Client
from langsmith.schemas import ListPromptsResponse

#### LangSmith API keys

In [4]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "LangSmith_Fundamentos"

In [5]:
# Ingresa el API key - Gemini
#os.environ["GOOGLE_API_KEY"]: str = getpass.getpass("Ingresa tu Google AI API key: ")

In [6]:

# Ingresa el API key - LangSmith
#os.environ["LANGSMITH_API_KEY"]: str = getpass.getpass("Ingresa tu Google AI API key: ")

In [7]:
#Inicializamos nuestro modelo chat
llm: ChatGoogleGenerativeAI = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

####**Gestión de prompts en LangSmith**

In [8]:
#Inicializamos el cliente LangSmith
client = Client()

In [9]:
#Creamos el prompt template
mensaje_sistema: str = 'Eres un experto en geografía e historia. Cuando te pregunten por una capital, responde con la capital y agrega un breve contexto histórico relevante.'
mensaje_humano: str = '¿Cuál es la capital de {país}?'

prompt_template: ChatPromptTemplate = ChatPromptTemplate([
    ("system", mensaje_sistema),
    ("user", mensaje_humano)
])

#Guardamos el prompt en el repositorio
client.push_prompt("geografia", object=prompt_template, is_public=False)

'https://smith.langchain.com/prompts/geografia/65ab5da5?organizationId=12a7beac-1159-486d-a3bf-07c617d75cca'

In [10]:
#Guardar prompt con cofiguración del modelo
chain: RunnableSequence = prompt_template | llm

#Guardamos el prompt en el repositorio
client.push_prompt("geografia_chain", object=chain, is_public=False)

'https://smith.langchain.com/prompts/geografia_chain/ea62634b?organizationId=12a7beac-1159-486d-a3bf-07c617d75cca'

In [11]:
#Leemos el prompt del repositorio
prompt: ChatPromptTemplate = client.pull_prompt("geografia", include_model=True)

In [12]:
prompt_generado: ChatPromptValue = prompt.invoke({"país": "Colombia"})

for mensaje in prompt_generado.messages:
    print(f'{mensaje.type}: {mensaje.content}')

system: Eres un experto en geografía e historia. Cuando te pregunten por una capital, responde con la capital y agrega un breve contexto histórico relevante.
human: ¿Cuál es la capital de Colombia?


In [13]:
#Leemos el prompt del repositorio

prompt: RunnableSequence = client.pull_prompt("geografia_chain", include_model=True, secrets_from_env=True)

print(prompt)

first=ChatPromptTemplate(input_variables=['país'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'geografia_chain', 'lc_hub_commit_hash': 'ea62634b10ba3db0de2cf5db79347a98c6414b38d8ddb67430d10bbf4a4ecd88'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Eres un experto en geografía e historia. Cuando te pregunten por una capital, responde con la capital y agrega un breve contexto histórico relevante.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['país'], input_types={}, partial_variables={}, template='¿Cuál es la capital de {país}?'), additional_kwargs={})]) middle=[] last=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain-google-genai': '4.2.6'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, '

In [14]:
respuesta: AIMessage = prompt.invoke({"país": "Colombia"})

print(respuesta.content)

La capital de Colombia es **Bogotá**.

Fundada el 6 de agosto de 1538 por el conquistador español Gonzalo Jiménez de Quesada, fue establecida como la capital del Nuevo Reino de Granada, el virreinato español en la región. Desde entonces, ha sido un centro político, económico y cultural fundamental, desempeñando un papel crucial en la independencia de Colombia y en la configuración de la nación.


In [18]:
#Listar prompts
prompts: ListPromptsResponse = client.list_prompts(query="geografia", is_public=False)

for repo in prompts.repos:
  print(repo.full_name)

Failed to refresh cache entry geografia_chain:with_model: Resource not found for /commits/-/geografia_chain/latest?include_model=true. HTTPError('404 Client Error: Not Found for url: https://api.smith.langchain.com/commits/-/geografia_chain/latest?include_model=true', '{"error":"Commit not found"}\n')
Failed to refresh cache entry geografia_chain:with_model: Resource not found for /commits/-/geografia_chain/latest?include_model=true. HTTPError('404 Client Error: Not Found for url: https://api.smith.langchain.com/commits/-/geografia_chain/latest?include_model=true', '{"error":"Commit not found"}\n')
Failed to refresh cache entry geografia_chain:with_model: Resource not found for /commits/-/geografia_chain/latest?include_model=true. HTTPError('404 Client Error: Not Found for url: https://api.smith.langchain.com/commits/-/geografia_chain/latest?include_model=true', '{"error":"Commit not found"}\n')
Failed to refresh cache entry geografia_chain:with_model: Resource not found for /commits/-

In [17]:
#Eliminar prompt
client.delete_prompt("geografia")

client.delete_prompt("geografia_chain")